# SpendShield v2 research explanation evaluation

This experiment reviews deterministic explanations generated from observed v2 feature values. Explanations do not read synthetic labels, generator rules, future records, or human decisions. The synthetic scenario label appears only as an offline audit column in the bounded sample.

In [1]:
from __future__ import annotations

import csv
import sys
import json
from pathlib import Path

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / 'ml').exists():
        ROOT = candidate
        break
sys.path.insert(0, str(ROOT))
ARTIFACT_DIR = ROOT / 'data' / 'synthetic' / 'anomaly_detection' / 'v2'
FEATURE_MANIFEST = ROOT / 'data' / 'synthetic' / 'v2' / 'features' / 'feature_manifest.json'
from ml.anomaly_detection import MODEL_FEATURE_NAMES
from ml.explainability import FORBIDDEN_EXPLANATION_TERMS, validate_explanations

summary = json.loads((ARTIFACT_DIR / 'anomaly_evaluation_summary.json').read_text(encoding='utf-8'))
synthetic_labels = json.loads(FEATURE_MANIFEST.read_text(encoding='utf-8'))['target']['classes']
validation = json.loads((ARTIFACT_DIR / 'explanation_validation.json').read_text(encoding='utf-8'))
with (ARTIFACT_DIR / 'explanation_samples.csv').open('r', encoding='utf-8', newline='') as handle:
    samples = list(csv.DictReader(handle))
len(samples), validation['valid']

(20, True)

## Explanation contract

Each explanation names actual available features, reports signal strength, warns when prior history is limited, and uses cautious research language. It must not say that a transaction is fraudulent, must not expose scenario labels in the text, and must not claim causality or certainty.

In [2]:
explanation_records = []
for row in samples:
    explanation_records.append({
        'synthetic_transaction_id': row['synthetic_transaction_id'],
        'signal_feature_names': json.loads(row['signal_feature_names']),
        'explanation_text': row['explanation_text'],
        'missing_history_warning': row['missing_history_warning'].lower() == 'true',
    })
manual_review_sample = explanation_records[:5]
manual_review_sample

[{'synthetic_transaction_id': 'syn-transaction-009080',
  'signal_feature_names': ['amount',
   'channel_novelty_before',
   'merchant_novelty_before',
   'transaction_hour',
   'user_historical_average_amount_before',
   'user_relative_amount_deviation',
   'user_relative_time_deviation'],
  'explanation_text': "Research explanation: The amount is 4.17x the user's prior observed average (979.08 to 4083.39); this is a strong amount-deviation signal. The user's observed merchant and channel history is new at this row; this is a strong novelty signal. The transaction hour differs from the user's prior observed timing by 2.50 circular hours; this is a moderate time-deviation signal. Multiple observed signals occurred together and jointly raised the research anomaly score.",
  'missing_history_warning': False},
 {'synthetic_transaction_id': 'syn-transaction-008917',
  'signal_feature_names': ['amount',
   'transaction_hour',
   'user_historical_average_amount_before',
   'user_relative_amo

In [3]:
checked = validate_explanations(
    explanation_records,
    available_features=MODEL_FEATURE_NAMES,
    synthetic_labels=synthetic_labels,
    max_rows=25,
)
{
    'validation_file_valid': validation['valid'],
    'revalidated': checked,
    'forbidden_terms_checked': list(FORBIDDEN_EXPLANATION_TERMS),
    'synthetic_label_evaluation_only': True,
}

{'validation_file_valid': True,
 'revalidated': {'valid': True,
  'errors': [],
  'warnings': [],
  'rows_checked': 20,
  'bounded_sample_limit': 25,
  'forbidden_terms_checked': ['fraud',
   'financial crime',
   'payment blocking',
   'payment rejection',
   'confirmed fraud',
   'guaranteed fraud',
   'automatic fraud decision'],
  'labels_used_for_validation_only': True},
 'forbidden_terms_checked': ['fraud',
  'financial crime',
  'payment blocking',
  'payment rejection',
  'confirmed fraud',
  'guaranteed fraud',
  'automatic fraud decision'],
 'synthetic_label_evaluation_only': True}

## Limitations and decision

These are research explanations, not human investigator findings. The sample is bounded for manual review, signal thresholds are fixed research assumptions, and synthetic labels are used only for offline audit. No production inference or financial action is connected to the output.

In [4]:
{
    'sample_count': len(samples),
    'deterministic_output': validation['deterministic_output'],
    'decision': 'ANOMALY_EVALUATION_READY_FOR_BACKEND_RESEARCH_INTEGRATION' if checked['valid'] and validation['deterministic_output'] else 'ANOMALY_EVALUATION_REQUIRES_REVIEW',
    'production_inference_created': False,
}

{'sample_count': 20,
 'deterministic_output': True,
 'decision': 'ANOMALY_EVALUATION_READY_FOR_BACKEND_RESEARCH_INTEGRATION',
 'production_inference_created': False}